In [ ]:
# --- 修复部分开始 ---
# 1. 强制安装 zstd 解压工具 (修复报错的关键)
!sudo apt-get update && sudo apt-get install -y zstd
# --- 修复部分结束 ---

# 2. 安装 Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!pip install aiohttp gradio

import os
import time
import subprocess
import threading

# 3. 在后台启动 Ollama 服务
def start_ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "serve"])

ollama_thread = threading.Thread(target=start_ollama)
ollama_thread.start()

# 等待几秒钟让服务启动
print("正在启动 Ollama 服务，请稍候...")
time.sleep(10)

# 4. 下载模型 (如果之前没下载成功)
print("正在下载 Llama 3.1 模型...")
!ollama run llama3.1 "你好"

# 5. 编写 Chatbot 界面代码
import gradio as gr
import json
import requests

def chat_response(message, history):
    url = "http://localhost:11434/api/chat"
    formatted_history = []
    for user_msg, ai_msg in history:
        formatted_history.append({"role": "user", "content": user_msg})
        formatted_history.append({"role": "assistant", "content": ai_msg})
    formatted_history.append({"role": "user", "content": message})

    payload = {
        "model": "llama3.1",
        "messages": formatted_history,
        "stream": True
    }

    response = requests.post(url, json=payload, stream=True)

    partial_message = ""
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode('utf-8')
            json_line = json.loads(decoded_line)
            if 'message' in json_line:
                content = json_line['message'].get('content', '')
                partial_message += content
                yield partial_message

print("正在启动聊天界面...")
demo = gr.ChatInterface(
    chat_response,
    title="我的 Colab Llama 机器人",
    description="运行在 Google Colab T4 GPU 上的 Llama 3.1",
)

demo.launch(share=True, debug=True)

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
0% [Working]^C
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#############                                                             19.3%